# Pip 3D Model Generation via TripoSR

This notebook converts a 2D reference image of Pip the fox into a 3D model using **TripoSR**, an open-source image-to-3D model by Stability AI and Tripo.

**Runtime requirement:** This notebook MUST run on a GPU. Before starting:
1. Click **Runtime** → **Change runtime type**
2. Select **GPU** (e.g., T4 or V100)
3. Click **Save**

Then run all cells in order.

## Step 1: Install Dependencies

This cell installs TripoSR and all required packages. It may take 3–5 minutes. If TripoSR fails, it will fallback to Stable Fast 3D.

In [ ]:
import subprocess
import sys

print("Installing base dependencies...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pillow einops omegaconf rembg

print("\nAttempting to install TripoSR...")
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 
         "git+https://github.com/VAST-AI-Research/TripoSR.git@main"],
        capture_output=True,
        timeout=300,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    print("✓ TripoSR installed successfully")
    USING_TRIPSR = True
except Exception as e:
    print(f"⚠ TripoSR installation failed: {e}")
    print("\nFalling back to Stable Fast 3D...")
    try:
        !pip install -q git+https://github.com/stable-x/stable-fast-3d.git
        print("✓ Stable Fast 3D installed")
        USING_TRIPSR = False
    except:
        print("⚠ Both models failed to install. Check GPU memory and restart runtime.")
        USING_TRIPSR = None

print("\n✓ Dependencies installed successfully")
print(f"  Using: {'TripoSR' if USING_TRIPSR else 'Stable Fast 3D' if USING_TRIPSR is False else 'ERROR'}")

## Step 2: Import Libraries and Set Up Device

Verify GPU is available and load the TripoSR model.

In [ ]:
import torch
from PIL import Image
import numpy as np
from pathlib import Path

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("GPU not available! Please change runtime to GPU in Runtime > Change runtime type")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\n✓ Using device: {device}")

## Step 3: Load TripoSR Model

Load the pre-trained TripoSR model (may take 1–2 minutes on first run).

In [ ]:
print("Loading 3D generation model...")

if USING_TRIPSR:
    from tripo_sr.models import TripoSRModel
    model = TripoSRModel.from_pretrained_huggingface()
    model.to(device)
    model.eval()
    print("✓ TripoSR model loaded")
elif USING_TRIPSR is False:
    # Stable Fast 3D
    from sf3d.utils import preprocess_image
    from sf3d.models import TripoSR
    import torch
    model = TripoSR()
    model = model.to(device)
    model.eval()
    print("✓ Stable Fast 3D model loaded")
else:
    raise RuntimeError("No model loaded. Check Step 1 output and restart runtime.")

## Step 4: Upload Reference Image

Upload the 2D reference image. It should be a clean, well-lit image of Pip the fox against a simple background.

**Supported formats:** PNG, JPG, JPEG, WEBP

In [ ]:
from google.colab import files
import os

print("Upload your reference image (click 'Choose Files' below):")
uploaded = files.upload()

# Get the uploaded filename
image_path = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {image_path}")
print(f"  File size: {os.path.getsize(image_path) / 1e6:.2f} MB")

# Display the image
img = Image.open(image_path)
print(f"  Dimensions: {img.size}")
img.thumbnail((400, 400))
img.show()

## Step 5: Run TripoSR Inference

Generate the 3D mesh from the 2D image. This typically takes 1–3 minutes.

In [ ]:
from PIL import Image
import torch

print("Preprocessing image...")
# Load image
image = Image.open(image_path).convert('RGB')

# Remove background (optional but recommended for better 3D results)
print("  Removing background...")
from rembg import remove
image_rembg = remove(image)

# Resize to model input size
print("  Resizing foreground...")
if USING_TRIPSR:
    from tripo_sr.utils import resize_foreground
    image_processed = resize_foreground(image_rembg, 0.85)
else:
    # Stable Fast 3D preprocessing
    image_processed = image_rembg.resize((320, 320))

print("\nRunning inference (this may take 2–4 minutes)...")
with torch.no_grad():
    if USING_TRIPSR:
        mesh = model(image_processed, quality="medium")
    else:
        # Stable Fast 3D inference
        import numpy as np
        img_array = np.array(image_processed) / 255.0
        mesh = model.generate_mesh(img_array)

print("✓ Inference complete!")
print(f"  Mesh vertices: {len(mesh.vertices)}")
print(f"  Mesh faces: {len(mesh.faces)}")

## Step 6: Export as GLB

Save the 3D mesh as a `.glb` file (binary GLTF format, suitable for Mixamo and other 3D tools).

In [ ]:
output_path = "pip_model.glb"
print(f"Exporting mesh to {output_path}...")
mesh.export(output_path)
file_size = os.path.getsize(output_path) / 1e6
print(f"✓ Exported successfully ({file_size:.2f} MB)")

## Step 7: Download the 3D Model

Download the generated `.glb` file to your machine. Save it to:

```
pip-pipeline/assets/meshes/pip_model.glb
```

In [ ]:
print(f"Downloading {output_path}...")
files.download(output_path)
print("✓ Download started in your browser")
print(f"\nMove the downloaded file to: assets/meshes/pip_model.glb")
print("Then proceed to Stage 2 (prep_for_mixamo.py)")

## Troubleshooting

**Issue: "ModuleNotFoundError: No module named 'tripo_sr'"**
- This notebook automatically falls back to **Stable Fast 3D** if TripoSR fails
- Check Step 1 output to see which model is being used
- Restart runtime (Runtime > Restart runtime) and re-run all cells

**Issue: "CUDA out of memory" error**
- Restart the runtime (Runtime > Restart runtime)
- Try with lower resolution or quality settings
- Or switch to a free T4 GPU if you have a choice

**Issue: Background removal looks bad**
- Comment out the `from rembg import remove` line in Step 5
- Or pre-process the image background before uploading

**Issue: Both installations failed**
- Ensure GPU is enabled (Runtime > Change runtime type > GPU)
- Check your Colab session hasn't hit resource limits
- Try in a fresh Colab notebook window

**Model differences:**
- **TripoSR:** Faster, often better detail (preferred if available)
- **Stable Fast 3D:** More stable installation, good quality (fallback option)
- Both produce compatible GLB files for Mixamo